# 03b · Familia State of the Art: LLM por prompting

**Planteamiento:** cada cliente se convierte en texto (cupo, edad, educación, historial de 6 meses de estado, saldo y pago, y agregados). Al LLM se le pide una **probabilidad de impago en octubre**, con unos pocos ejemplos etiquetados del propio entrenamiento (*few-shot*), temperatura 0 y respuesta en JSON.

**Modelo:** un LLM abierto que corre **localmente con Ollama** (por defecto `qwen2.5:3b`). No necesita API key ni cuenta, los datos de los clientes no salen de la máquina y el resultado es reproducible con la misma versión del modelo. Si se define `ANTHROPIC_API_KEY`, usa Claude.

**Evaluación justa:**
- **Muestra:** 200 clientes estratificados del **entrenamiento**. El hold-out no se toca.
- **Comparación:** sobre esos mismos 200 clientes, la AUC del LLM contra la del mejor tabular, calculada con predicciones *out-of-fold*.
- **Por qué no la CV completa:** cada cliente es una inferencia de unos segundos, así que la CV con 24.000 filas no es viable. La incertidumbre se reporta con bootstrap.

Este notebook agrega **una sola corrida** (fase `candidato`, familia `sota_llm`) al experimento del notebook 03; no repite las demás.

In [1]:
import sys, os, re, json, time, warnings
sys.path.insert(0, '..'); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, mlflow, requests
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, cross_val_predict
from src.datos import cargar_obt, dividir, esquema_cv, SEMILLA
from src.pipeline import construir_pipeline
from src import tracking
tracking.configurar(); exp = mlflow.get_experiment_by_name(tracking.EXPERIMENTO)
obt, meta = cargar_obt(); X_train, X_test, y_train, y_test = dividir(obt)
OLLAMA = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
PROVEEDOR = 'anthropic' if os.environ.get('ANTHROPIC_API_KEY') else 'ollama'
MODELO_LLM = os.environ.get('LLM_MODEL', 'claude-haiku-5-5' if PROVEEDOR == 'anthropic' else 'qwen2.5:3b')
print('Proveedor:', PROVEEDOR, '| modelo:', MODELO_LLM)
if PROVEEDOR == 'ollama':
    disponibles = [m['name'] for m in requests.get(f'{OLLAMA}/api/tags', timeout=10).json()['models']]
    print('Modelos en Ollama:', disponibles)
    assert any(m.startswith(MODELO_LLM) for m in disponibles), f'Falta el modelo: ollama pull {MODELO_LLM}'

Proveedor: ollama | modelo: qwen2.5:3b
Modelos en Ollama: ['qwen2.5:3b']


## Prompt

In [2]:
MESES = ['sep', 'ago', 'jul', 'jun', 'may', 'abr']
def describir(r):
    hist = '; '.join(f"{m}: estado {int(r[f'pay_status_m{i}'])}, saldo {r[f'bill_amt_m{i}']:.0f}, pago {r[f'pay_amt_m{i}']:.0f}"
                     for i, m in zip(range(1, 7), MESES))
    return (f"cupo {r.limit_bal:.0f}, edad {int(r.age)}, máximo atraso en 6 meses {int(r.max_delay_6m)}, "
            f"meses en mora {int(r.n_delinquent_months_6m)}, utilización media {r.avg_utilization_6m:.2f}. Historial: {hist}")

SISTEMA = ('Eres analista de riesgo de tarjetas de crédito (Taiwán, 2005, montos en NT$). Estima la probabilidad de que el cliente '
           'NO pague su cuota de octubre. Estados de pago: -2 sin consumo, -1 pagó el total, 0 pagó el mínimo, k = k meses de atraso. '
           'La tasa base de impago es 22%. Responde SOLO con JSON: {"p": número entre 0 y 1}.')

idx_resto, idx_muestra = train_test_split(np.arange(len(X_train)), test_size=200, stratify=y_train, random_state=SEMILLA)
y_resto = y_train.iloc[idx_resto]
shots = pd.concat([X_train.iloc[idx_resto][y_resto.values == c].sample(2, random_state=SEMILLA) for c in (0, 1)]).sample(frac=1, random_state=SEMILLA)
EJEMPLOS = '\n\n'.join(f"Cliente: {describir(r)}\nRespuesta: {{\"p\": {0.8 if y_train.loc[i] else 0.1}}}" for i, r in shots.iterrows())
mensaje = lambda r: f"Ejemplos:\n{EJEMPLOS}\n\nCliente: {describir(r)}\nRespuesta:"
print(SISTEMA + '\n\n' + mensaje(X_train.iloc[idx_muestra[0]]))

Eres analista de riesgo de tarjetas de crédito (Taiwán, 2005, montos en NT$). Estima la probabilidad de que el cliente NO pague su cuota de octubre. Estados de pago: -2 sin consumo, -1 pagó el total, 0 pagó el mínimo, k = k meses de atraso. La tasa base de impago es 22%. Responde SOLO con JSON: {"p": número entre 0 y 1}.

Ejemplos:
Cliente: cupo 1000000, edad 47, máximo atraso en 6 meses 0, meses en mora 0, utilización media 0.88. Historial: sep: estado 0, saldo 964511, pago 50784; ago: estado 0, saldo 983931, pago 50723; jul: estado 0, saldo 535020, pago 896040; jun: estado -1, saldo 891586, pago 50000; may: estado 0, saldo 927171, pago 50000; abr: estado 0, saldo 961664, pago 50256
Respuesta: {"p": 0.1}

Cliente: cupo 430000, edad 42, máximo atraso en 6 meses 2, meses en mora 6, utilización media 0.70. Historial: sep: estado 2, saldo 289358, pago 13000; ago: estado 2, saldo 296219, pago 11000; jul: estado 2, saldo 300873, pago 0; jun: estado 2, saldo 294527, pago 19000; may: estado 2

## Inferencia

In [3]:
def extraer_p(texto):
    try:
        return float(np.clip(float(json.loads(texto)['p']), 0, 1))
    except Exception:
        m = re.search(r'(0(?:\.\d+)?|1(?:\.0+)?)', texto)
        return float(m.group(1)) if m else np.nan

def preguntar(r):
    for intento in range(3):
        try:
            if PROVEEDOR == 'ollama':
                resp = requests.post(f'{OLLAMA}/api/chat', timeout=120, json={
                    'model': MODELO_LLM, 'stream': False, 'format': 'json',
                    'options': {'temperature': 0, 'seed': SEMILLA, 'num_predict': 20},
                    'messages': [{'role': 'system', 'content': SISTEMA}, {'role': 'user', 'content': mensaje(r)}]})
                return extraer_p(resp.json()['message']['content'])
            import anthropic
            m = anthropic.Anthropic().messages.create(model=MODELO_LLM, max_tokens=20, temperature=0, system=SISTEMA,
                                                      messages=[{'role': 'user', 'content': mensaje(r)}])
            return extraer_p(m.content[0].text)
        except Exception as e:
            time.sleep(2 * (intento + 1))
    return np.nan

muestra = X_train.iloc[idx_muestra]; ys = y_train.iloc[idx_muestra].values
t0 = time.perf_counter(); p_llm = []
for k, (_, r) in enumerate(muestra.iterrows()):
    p_llm.append(preguntar(r))
    if (k + 1) % 50 == 0: print(f'{k+1}/200 · {time.perf_counter()-t0:.0f}s')
p_llm = np.array(p_llm); seg = time.perf_counter() - t0; ok = ~np.isnan(p_llm)
print(f'Respuestas válidas: {ok.mean():.0%} | {seg/len(muestra):.2f} s por cliente | valores distintos: {len(np.unique(p_llm[ok]))}')

50/200 · 40s


100/200 · 75s


150/200 · 111s


200/200 · 146s
Respuestas válidas: 100% | 0.73 s por cliente | valores distintos: 8


## Comparación con el mejor modelo tabular sobre la misma muestra

In [4]:
cand = mlflow.search_runs([exp.experiment_id], filter_string="tags.fase = 'candidato'", order_by=['metrics.cv_roc_auc_mean DESC'])
cand = cand[cand['tags.familia'] != 'sota_llm']; mejor = cand.iloc[0]
pipe_mejor = mlflow.sklearn.load_model(f"runs:/{mejor.run_id}/modelo")
oof = cross_val_predict(pipe_mejor, X_train, y_train, cv=esquema_cv(), method='predict_proba')[:, 1]
auc_llm = roc_auc_score(ys[ok], p_llm[ok]); auc_tab = roc_auc_score(ys[ok], oof[idx_muestra][ok])
rng = np.random.RandomState(SEMILLA); boot = []
for _ in range(500):
    b = rng.choice(ok.sum(), ok.sum())
    if len(set(ys[ok][b])) == 2: boot.append(roc_auc_score(ys[ok][b], p_llm[ok][b]))
regla = (muestra.pay_status_m1.values >= 1).astype(float)
auc_regla = roc_auc_score(ys, regla)
res = pd.DataFrame({'AUC en la muestra (n=200)': [auc_llm, auc_tab, auc_regla]},
                   index=[f'LLM {MODELO_LLM}', f"{mejor['tags.modelo']} (OOF)", 'regla de negocio'])
print(f'LLM: AUC {auc_llm:.3f} (IC95% bootstrap {np.percentile(boot,2.5):.3f}–{np.percentile(boot,97.5):.3f})')
res.round(3)

LLM: AUC 0.593 (IC95% bootstrap 0.490–0.690)


,AUC en la muestra (n=200)
LLM qwen2.5:3b,0.593
hist_gradient_boosting (OOF),0.815
regla de negocio,0.655


In [5]:
with mlflow.start_run(run_name='llm_prompting') as run:
    mlflow.set_tags({'fase': 'candidato', 'familia': 'sota_llm', 'modelo': 'llm_prompting', 'git_commit': tracking.commit_git(), 'proveedor': PROVEEDOR})
    tracking.log_datos(meta)
    mlflow.log_params({'llm_modelo': MODELO_LLM, 'semilla': SEMILLA, 'n_muestra': int(ok.sum()), 'few_shot': len(shots),
                       'temperatura': 0, 'evaluacion': 'muestra estratificada de 200 clientes de entrenamiento'})
    mlflow.log_metrics({'cv_roc_auc_mean': auc_llm, 'cv_roc_auc_std': float(np.std(boot)), 'auc_tabular_misma_muestra': auc_tab,
                        'auc_regla_misma_muestra': auc_regla, 'segundos_por_cliente': seg / len(muestra),
                        'respuestas_validas': float(ok.mean()), 'cv_seconds_total': seg})
    mlflow.log_text(SISTEMA + '\n\n' + EJEMPLOS, 'prompt.txt')
    pd.DataFrame({'customer_id': obt.loc[muestra.index, 'customer_id'] if 'customer_id' in obt else muestra.index,
                  'y': ys, 'p_llm': p_llm, 'p_tabular_oof': oof[idx_muestra]}).to_csv('../docs/llm_muestra.csv', index=False)
    mlflow.log_artifact('../docs/llm_muestra.csv')
print('Corrida registrada:', run.info.run_id)

2026/11/18 21:00:44 INFO mlflow.tracking._tracking_service.client: 🏃 View run llm_prompting at: http://localhost:5001/#/experiments/1/runs/37786ad811394aa8a25a11f2f6971661.


2026/11/18 21:00:44 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5001/#/experiments/1.


Corrida registrada: 37786ad811394aa8a25a11f2f6971661


**Lectura:** el LLM se compara con el mejor tabular y con la regla de negocio sobre exactamente los mismos 200 clientes. Aunque llegara a igualar al tabular, **no es candidato a campeón**:
- **Costo y latencia:** una inferencia de segundos por cliente, frente a microsegundos.
- **Calibración:** sus probabilidades son pocas y redondeadas, así que la curva ROC queda escalonada.
- **Escala y comparación:** no admite la misma CV de 24.000 filas, y sobre 200 clientes la incertidumbre es ±0,04 de AUC.

El experimento muestra que, para datos tabulares con señal numérica, un modelo de boosting entrenado específicamente sigue siendo mejor y mucho más barato que un LLM genérico.